# Khai báo thư viện

In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.5 MB/s eta 0:00:00


In [2]:
import os
import yaml
from google.colab import drive
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Mount Drive và Ánh xạ cấu trúc dataset

In [3]:
drive.mount('/content/drive')

# đường dẫn tới dataset
dataset_root = '/content/drive/MyDrive/SelfProject/HDIRLT/datasource'
yaml_file_path = os.path.join(dataset_root, 'data.yaml') # đường dẫn tuyệt đối của file yaml

# Phân tích file cấu hình
with open(yaml_file_path, 'r') as file:
    yaml_data = yaml.load(file, Loader=yaml.FullLoader) # trả về dictionary vì bản chất của yaml là các cặp key - value

# Trích xuất thông tin
print("Label Space: ")
print(yaml_data['names'])
print(f"Số lượng class: {yaml_data['nc']}")

Mounted at /content/drive
Label Space: 
['bike', 'helmet', 'lisc', 'no_helmet', 'noise', 'rider']
Số lượng class: 6


# Khởi tạo không gian tham số của mô hình

In [ ]:
model_weights = 'yolo11m.pt' # phiên bản mô hình

# khởi tạo không gian tham số của mạng nơ ron
model = YOLO(model_weights)

# Trích xuất và in ra bảng tóm tắt kiến trúc
# Bao gồm số lượng layer, số lượng parameters, và khối lượng tính toán
model.info()

YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.5 GFLOPs


(231, 20114688, 0, 68.52838399999999)

Các chỉ số Parameters và GFLOPs cho ta biết thông tin về việc mô hình sẽ sử dụng RAM/VRAM và GPU/NPU như thế nào.
Với GFLOPs quá cao, chip sẽ dễ dàng dính bottleneck threshold.

# Thiết lập huấn luyện

In [ ]:
results = model.train(
    data = yaml_file_path,
    epochs = 10,
    imgsz = 640,
    batch = 8,
    project = "/content/drive/MyDrive/SelfProject/HDIRLT/Helmet_Detection",
    name = 'baseline_6_classes',
    device = '0',
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SelfProject/HDIRLT/datasource, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_6_classes-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

Thực nghiệm 1:
Tăng batch size lên 32

In [ ]:
# khởi tạo không gian tham số của mạng nơ ron
model = YOLO("yolo11m.pt")


results = model.train(
    data = yaml_file_path,
    epochs = 10,
    imgsz = 640,
    batch = 16,
    project = "/content/drive/MyDrive/SelfProject/HDIRLT/Helmet_Detection",
    name = 'baseline_6_classes_exp1',
    device = '0',
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SelfProject/HDIRLT/datasource/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_6_classes_exp1, nbs=64, nms=False, opset=None, optimize=False, 

Thực nghiệm 2: tăng batch size lên 32

In [6]:
# Thực nghiệm tối ưu hóa với Batch Size lớn và Epochs dài hơn
model_pro_exp = YOLO('yolo11m.pt')

results_pro = model_pro_exp.train(
    data=yaml_file_path,
    epochs=50,          # Tăng số epoch để mô hình hội tụ sâu
    imgsz=640,
    batch=32,           # Với Colab Pro/A100, batch 32 sẽ chạy mượt
    project="/content/drive/MyDrive/SelfProject/HDIRLT/Helmet_Detection",
    name='pro_batch32_epochs50',
    device='0',         # Nếu Colab Pro cho A100, GPU vẫn thường là '0'
    patience=10         # Dừng sớm nếu không còn cải thiện sau 10 epoch
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SelfProject/HDIRLT/datasource/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pro_batch32_epochs50-2, nbs=64, nms=False, opset=None, opti

In [8]:
metrics = model_pro_exp.val(
    data = yaml_file_path,
    split = 'test',
    conf=0.25, # trên conf % thì sẽ xác định là object
    iou=0.6 # mức độ trùng khớp giữa bboxx dự đoán và bbox thực tế
)

# Trích xuất các thông số toán học từ object metrics
'''
Các metric khác nhau ở ngưỡng IoU được sử dụng để xác định một dự đoán là True Positive hay False Positive
'''
print("---KẾT QUẢ TỔNG THỂ ---")
print(f"mAP: {metrics.box.map}")
print(f"mAP50: {metrics.box.map50}") # IoU > 0.5
print(f"mAP75: {metrics.box.map75}") # IoU > 0.75
print(f"mAP@50-95: {metrics.box.map}") # Đây là điểm trung bình cộng của 10 giá trị mAP tại các ngưỡng IoU trải đều từ $0.5$ đến $0.95$ với bước nhảy $0.05$

class_index_no_helmet = 3
print("\n--- KẾT QUẢ TRỌNG TÂM ---")
print(f"mAP@50 (no_helmet): {metrics.box.maps[class_index_no_helmet]}")

# Trích xuất Recall (Độ phủ) cho class no_helmet
print(f"Recall (no_helmet): {metrics.box.r[class_index_no_helmet]}")

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,034,658 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 1.1±1.1 ms, read: 0.1±0.0 MB/s, size: 82.9 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1lTNOjkOaQRMQjP39icqCgHDfBa1xZiKg/SelfProject/HDIRLT/datasource/test/labels... 260 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 260/260 9.8it/s 26.4s
val: New cache created: /content/drive/.shortcut-targets-by-id/1lTNOjkOaQRMQjP39icqCgHDfBa1xZiKg/SelfProject/HDIRLT/datasource/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 4.3it/s 4.0s
                   all        260       2992      0.866      0.899      0.883       0.68
                  bike        252        702        0.9      0.927      0.945      0.822
                helmet        197        379      0.866      0.905      0.874   

thực nghiệm

In [ ]:
# # Tải lại trọng số tốt nhất sau quá trình train (YOLO lưu tại thư mục weights/best.pt)
# best_model_path = '/content/drive/MyDrive/_____/Helmet_Detection/baseline_6_classes/weights/best.pt'
# model_inference = YOLO(best_model_path)

# # Thực thi mô hình trên dữ liệu thực tế
# results_inference = model_inference._____(
#     source='_____.mp4', # Đường dẫn tới file ảnh hoặc video test của bạn
#     conf=_____,        # Ngưỡng tự tin (Confidence Threshold)
#     iou=0.45,          # NMS Threshold
#     save=True,         # Lưu kết quả ảnh/video đã vẽ Bounding Box
#     show_labels=True,
#     show_conf=True
# )

# print(f"Video dự đoán đã được lưu tại: {results_inference[0].save_dir}")